# 15b — Merge eval-derived pairs into the curated stream

Feeds the 4,000-prompt eval feedback + gap-fill + reasoning (`thinking`) pairs
from `Train/eval_derived/` into `knowledge_pairs.jsonl` (tagged `NB15b`) so the
dataset assembler (NB16) picks them up → teacher (NB17) → distill (NB21) →
student booster (NB23). Idempotent: re-runs replace the NB15b rows.


In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import config

ED = config.TRAIN_ROOT / 'eval_derived'
FILES = ['ask_eval_pairs.jsonl', 'eval_feedback_pairs.jsonl', 'gapfill.jsonl', 'thinking.jsonl']

config.clean_notebook_pairs('NB15b')  # clean re-run, not duplicate
pairs = []
for name in FILES:
    p = ED / name
    if not p.exists():
        print('skip missing', name); continue
    kept = 0
    for line in p.read_text().splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if r.get('instruction') and r.get('output'):
            pairs.append({'instruction': r['instruction'], 'output': r['output'],
                          'category': r.get('category', 'code_generation'),
                          'task_type': r.get('task_type', 'code_generation'),
                          'source': r.get('source', 'eval_derived')})
            kept += 1
    print(f'  {name}: {kept}')

n = config.save_notebook_pairs('NB15b', pairs)
print(f'NB15b: merged {n}/{len(pairs)} eval-derived pairs into knowledge_pairs.jsonl')
